## 1. Types of calculations AiiDAlab provides


The adsorption workflow provides three types of calculations:

- **Pore analysis:** computes geometric properties (density, accessible surface area, accessible
  probe-occupiable volume and porosity) for a specified probe radius.

- **Test-particle insertion:** samples the insertion of a molecule
  into the framework to calculate the Henry coefficient and adsorption
  energetics at infinite dilution.

- **Grand Canonical Monte Carlo (GCMC):** samples adsorption at fixed
  temperature, volume and chemical potential, allowing the number of
  adsorbed molecules to fluctuate. Calculations at different pressures
  provide the adsorption isotherm.

## 2. Properties of the material

- Density: $0.576983\ \mathrm{g\,cm^{-3}}$
- Accessible surface area: $3964.65\ \mathrm{\AA^2}$
- Accessible probe-occupiable volume: $13737.4\ \mathrm{\AA^3}$
- Unit-cell volume: $17726.2\ \mathrm{\AA^3}$

The accessible porosity is calculated as

$$
\varepsilon
= \frac{V_{\mathrm{POAV}}}{V_{\mathrm{cell}}}
= \frac{13737.4}{17726.2}
= 0.77498.
$$

So, the accessible porosity is approximately $77.50\%$.

## 3. Henry coefficients

The computed Henry coefficients for $CO_2$ and $CH_4$ are $4.95566\cdot 10^{-6}$ $\frac{mol}{kg\cdot Pa}$ and $1.12067\cdot 10^{-6}$ $\frac{mol}{kg\cdot Pa}$, respectively.

Here, the loading is proportional to the gas partial
pressure: $q_i = K_{H,i}p_i$. The limiting adsorption selectivity is

$$
S_{\mathrm{CO_2/CH_4}}^{0}
= \frac{K_{H,\mathrm{CO_2}}}{K_{H,\mathrm{CH_4}}}
= 4.42
$$

IRMOF-1 thus preferentially adsorbs CO₂ at low pressure. 

## 4. Isotherms calculations


In [ ]:
import ast
import csv
import matplotlib.pyplot as plt

plt.rcdefaults()

FILES = [
    ("ch4_results.csv", r"$\mathrm{CH}_4$ isotherm"),
    ("co2_results.csv", r"$\mathrm{CO}_2$ isotherm"),
]

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
})

colors = [plt.colormaps["magma"](0.25), plt.colormaps["magma"](0.75)]
markers = ["o", "s"]

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

for (filename, label), color, marker in zip(FILES, colors, markers):
    with open(filename, newline="") as file:
        data = {
            row["Key"]: row["Value"]
            for row in csv.DictReader(file)
        }

    iso = ast.literal_eval(data["isotherm"])

    ax.errorbar(
        iso["pressure"],
        iso["loading_absolute_average"],
        yerr=iso["loading_absolute_dev"],
        label=label,
        color=color,
        marker=marker,
        linestyle="-",
        linewidth=1.6,
        markersize=5,
        capsize=3,
        elinewidth=1,
        capthick=1,
    )

ax.set(
    xlabel="Pressure (bar)",
    ylabel=r"Absolute loading (mol kg$^{-1}$)",
)

ax.legend(frameon=False)

ax.set_axisbelow(True)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()


# fig.savefig("isotherms_comparison.pdf", bbox_inches="tight")
# fig.savefig("isotherms_comparison.png", dpi=300, bbox_inches="tight")

plt.show()

Here $CO_2$ loading is higher than $CH_4$ loading throughout the pressure
range. The $CH_4$ isotherm is approximately linear, whereas the $CO_2$ isotherm
 is curved. Neither isotherm reaches a saturation plateau
within this pressure range.

## 5. Binary-mixture isotherms using IAST

Conditions : $T = 300\,\mathrm{K}$, $y_{\mathrm{CO_2}}=0.40$ and $y_{\mathrm{CH_4}}=0.60$ (biogas composition).

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyiast

isotherms = []
for filename in ["co2_results.csv", "ch4_results.csv"]:
    table = pd.read_csv(filename, index_col="Key")
    data = ast.literal_eval(table.loc["isotherm", "Value"]) #converts the dictionary stored as text into a python dictionary
    
    pure_gas = pd.DataFrame({"pressure": data["pressure"], "loading": data["loading_absolute_average"],})
    
    isotherm = pyiast.InterpolatorIsotherm(pure_gas, pressure_key="pressure", loading_key="loading") #calls the IAST intepolator which basically connects the puregas data points with straight lines
    isotherms.append(isotherm)

pressures = np.linspace(0.1, 3.9, 20)
pressures = np.append(pressures, 4.0) #add the last pressure point since the prevoius one ends with 3.9
y = np.array([0.40, 0.60]) #store the given gas fractions

co2_loadings = []
ch4_loadings = []
for pressure in pressures:
    partial_pressures = pressure * y
    co2_loading, ch4_loading = pyiast.iast(partial_pressures, isotherms) #call the IAST function which calculates the loadings based on the partial pressures and isotherms
    co2_loadings.append(co2_loading)
    ch4_loadings.append(ch4_loading)

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 12,
})

plt.figure(figsize=(6, 4), dpi=150)
plt.plot(pressures, co2_loadings, "o-", markersize=4, label="CO₂")
plt.plot(pressures, ch4_loadings, "s-", markersize=4, label="CH₄")
plt.xlabel("Total pressure (bar)")
plt.ylabel("Mixture loading (mol/kg)")
plt.title(r"$T = 300\,\mathrm{K}$, $y_{\mathrm{CO_2}} = 0.40$, $y_{\mathrm{CH_4}} = 0.60$")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()